# Feedforward Pointcloud Reconstruction

This tutorial runs **VGGT-X**, **MapAnything**, and **VGGT-Omega** on keyframes extracted from a real video, then compares the resulting pointclouds side by side. All three models run feedforward inference (no optimisation loop) and write results to zarr caches that downstream notebooks — `semantic_lifting`, `localization`, and `bundle_adjustment` — depend on. **Run this notebook before any of those.**

**Prerequisite:** [Keyframe Extraction](../01_preprocessing/keyframe_extraction.ipynb) must have been executed first to populate the `images/` directory used here.

In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
import os
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import open3d as o3d
import pyvista as pv
import torch
%matplotlib inline

# pv.set_jupyter_backend("static" if os.environ.get("PYVISTA_OFF_SCREEN") else "trame")
pv.set_jupyter_backend("trame")

from collab_splats.pointcloud.feedforward import VGGTXCreator, MapAnythingCreator, VGGTOmegaCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.pointcloud.utils import extrinsics_to_c2w
from collab_splats.utils.notebook import clean_and_extract_result, add_camera_frustums
from collab_splats.utils.visualization import (
    CAMERA_KWARGS,
    MESH_KWARGS,
    PCD_KWARGS,
    VIZ_KWARGS,
    pointcloud_to_polydata,
    visualize_splat,
)

In [3]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────

assert IMAGES.exists() and any(IMAGES.glob("*.jpg")), (
    f"No images found in {IMAGES}. Run 01_preprocessing/keyframe_extraction first."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}  |  images: {len(list(IMAGES.glob('*.jpg')))}")

device: cuda  |  images: 30


## §1 — VGGT-X Reconstruction

Loads a cached VGGT-X reconstruction from zarr if available, skipping GPU inference (~3–5 min on GPU). If no cache exists, runs VGGT-X on the keyframes and saves the result. The zarr file is chunked by frame for efficient per-frame random access by downstream notebooks (semantic_lifting, localization).

In [4]:
_vggtx_cache = CACHE_DIR / "vggtx" / "reconstruction.zarr"
_vggtx_cache.parent.mkdir(parents=True, exist_ok=True)

if _vggtx_cache.exists():
    result_vggt = FeedforwardResult.load_zarr(_vggtx_cache)
    print(f"Loaded VGGT-X result from cache  ({result_vggt.pts3d.shape[0]:,} pts)")
else:
    result_vggt = VGGTXCreator().run(IMAGES, device)
    result_vggt.save_zarr(_vggtx_cache)
    print(f"VGGT-X done  →  saved to {_vggtx_cache}")

[21:17:33] Loading model (cuda)...                                                                      ]8;id=7910187;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910188;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#574\574]8;;\

[21:18:50]   done in 77.9s                                                                              ]8;id=7910194;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910195;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#576\576]8;;\

           Preprocessing images...                                                                      ]8;id=7910201;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910202;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#580\580]8;;\

[21:18:53]   → 30 images  done in 3.0s                                                                  ]8;id=7910208;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910209;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#582\582]8;;\

[21:18:54] Running inference...                                                                         ]8;id=7910215;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910216;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#586\586]8;;\

[21:19:23]   done in 29.0s                                                                              ]8;id=7910222;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910223;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#591\591]8;;\

           Postprocessing...                                                                            ]8;id=7910229;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910230;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#595\595]8;;\

             → 500,000 pts  done in 0.8s                                                                ]8;id=7910236;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910237;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#598\598]8;;\

VGGT-X done  →  saved to /workspace/collab-splats/docs/source/.cache/birds_c0043/vggtx/reconstruction.zarr


## §2 — VGGT-X Post-processing and Visualisation

Removes statistical outliers and downsamples via voxel grid to produce a clean pointcloud. Confidence statistics are printed to help diagnose prediction quality.

In [5]:
pts3d_vggt, colors_vggt, conf_vggt_mean, conf_vggt_std = clean_and_extract_result(result_vggt, "VGGT-X")

[VGGT-X] Points: 500,000 raw → 171,665 filtered
[VGGT-X] Conf:   mean=2.334  std=1.276


## §3 — VGGT-X Pointcloud Viewer

Renders the filtered VGGT-X pointcloud with camera frustums overlaid. Blue frustums show predicted camera poses.

In [14]:
cloud_vggt = pointcloud_to_polydata(pts3d_vggt, RGB=colors_vggt)
pl = visualize_splat(
    cloud_vggt,
    aligned_cameras=extrinsics_to_c2w(result_vggt.extrinsics),
    mesh_kwargs=PCD_KWARGS,
    camera_kwargs=CAMERA_KWARGS,
    viz_kwargs=VIZ_KWARGS,
)
pl.show()

Widget(value='<iframe src="http://localhost:33539/index.html?ui=P_0x7172d8d32c90_3&reconnect=auto" class="pyvi…

## §4 — MapAnything Reconstruction

MapAnything predicts dense depth and camera poses via cross-view consistency.
Uses `confidence_percentile` to mask unreliable pixels before unprojection.

In [7]:
_ma_cache = CACHE_DIR / "mapanything" / "reconstruction.zarr"
_ma_cache.parent.mkdir(parents=True, exist_ok=True)

if _ma_cache.exists():
    result_ma = FeedforwardResult.load_zarr(_ma_cache)
    print(f"Loaded MapAnything result from cache  ({result_ma.pts3d.shape[0]:,} pts)")
else:
    result_ma = MapAnythingCreator().run(IMAGES, device)
    result_ma.save_zarr(_ma_cache)
    print(f"MapAnything done  →  saved to {_ma_cache}")

[21:19:29] Loading model (cuda)...                                                                      ]8;id=7910242;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910243;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#574\574]8;;\

Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /workspace/models/hub/facebookresearch_dinov2_main


[21:20:54]   done in 85.3s                                                                              ]8;id=7910248;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910249;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#576\576]8;;\

           Preprocessing images...                                                                      ]8;id=7910254;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910255;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#580\580]8;;\

...

[21:21:00]   done in 1.6s                                                                               ]8;id=7910280;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910281;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#591\591]8;;\

           Postprocessing...                                                                            ]8;id=7910286;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910287;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#595\595]8;;\

Checking triangle intersections: 100%|██████████| 1/1 [00:00<00:00, 303.65it/s]


[21:21:07]   → 500,000 pts  done in 7.1s                                                                ]8;id=7910292;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py\base.py]8;;\:]8;id=7910293;file:///workspace/collab-splats/collab_splats/pointcloud/feedforward/base.py#598\598]8;;\

MapAnything done  →  saved to /workspace/collab-splats/docs/source/.cache/birds_c0043/mapanything/reconstruction.zarr


## §5 — MapAnything Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as VGGT-X. Confidence metrics are collected for the side-by-side comparison table.

In [8]:
pts3d_ma, colors_ma, conf_ma_mean, conf_ma_std = clean_and_extract_result(result_ma, "MapAnything")

[MapAnything] Points: 500,000 raw → 467,019 filtered
[MapAnything] Conf:   mean=0.408  std=0.391


## §6 — MapAnything Pointcloud Viewer

Renders the filtered MapAnything pointcloud with camera frustums. Orange frustums show predicted camera poses; compare pose spread vs. VGGT-X above.

In [15]:
cloud_ma = pointcloud_to_polydata(pts3d_ma, RGB=colors_ma)
pl = visualize_splat(
    cloud_ma,
    aligned_cameras=extrinsics_to_c2w(result_ma.extrinsics),
    mesh_kwargs=PCD_KWARGS,
    camera_kwargs=CAMERA_KWARGS,
    viz_kwargs=VIZ_KWARGS,
)
pl.show()

Widget(value='<iframe src="http://localhost:33539/index.html?ui=P_0x7172d816bb50_4&reconnect=auto" class="pyvi…

## §7 — VGGT-Omega Reconstruction

VGGT-Omega jointly predicts camera poses and per-frame depth maps in a single forward pass. Loads from zarr cache if available, skipping GPU inference (~3–5 min on first run).

In [ ]:
_omega_cache = CACHE_DIR / "vggt_omega" / "reconstruction.zarr"
_omega_cache.parent.mkdir(parents=True, exist_ok=True)

if _omega_cache.exists():
    result_omega = FeedforwardResult.load_zarr(_omega_cache)
    print(f"Loaded VGGT-Omega result from cache  ({result_omega.pts3d.shape[0]:,} pts)")
else:
    result_omega = VGGTOmegaCreator().run(IMAGES, device)
    result_omega.save_zarr(_omega_cache)
    print(f"VGGT-Omega done  →  saved to {_omega_cache}")

## §8 — VGGT-Omega Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as §2 and §5 to the VGGT-Omega result.

In [ ]:
pts3d_omega, colors_omega, conf_omega_mean, conf_omega_std = clean_and_extract_result(result_omega, "VGGT-Omega")

In [ ]:
cloud_omega = pointcloud_to_polydata(pts3d_omega, RGB=colors_omega)
pl = visualize_splat(
    cloud_omega,
    aligned_cameras=extrinsics_to_c2w(result_omega.extrinsics),
    mesh_kwargs=PCD_KWARGS,
    camera_kwargs=CAMERA_KWARGS,
    viz_kwargs=VIZ_KWARGS,
)
pl.show()

## §9 — Three-way Comparison

Side-by-side PyVista viewer showing VGGT-X, MapAnything, and VGGT-Omega pointclouds with camera frustums overlaid. Blue = VGGT-X, orange = MapAnything, green = VGGT-Omega.

In [ ]:
pl = pv.Plotter(shape=(1, 3), window_size=(1800, 600))

pl.subplot(0, 0)
pl.add_mesh(cloud_vggt, **PCD_KWARGS)
add_camera_frustums(pl, result_vggt.extrinsics, color="cornflowerblue")
pl.add_text("VGGT-X", position="upper_left", font_size=14)

pl.subplot(0, 1)
pl.add_mesh(cloud_ma, **PCD_KWARGS)
add_camera_frustums(pl, result_ma.extrinsics, color="darkorange")
pl.add_text("MapAnything", position="upper_left", font_size=14)

pl.subplot(0, 2)
pl.add_mesh(cloud_omega, **PCD_KWARGS)
add_camera_frustums(pl, result_omega.extrinsics, color="mediumseagreen")
pl.add_text("VGGT-Omega", position="upper_left", font_size=14)

pl.show()

## §10 — Side-by-side Comparison

Tabulates raw vs. filtered point counts and confidence statistics for all three models. Higher confidence mean with lower std indicates more reliable depth predictions.

In [10]:
col = 14
print(f"{'Model':< {col}} {'Pts raw':>10} {'Pts filt':>10} {'Conf mean':>10} {'Conf std':>10}")
print("-" * (col + 42))
print(f"{'VGGT-X':< {col}} {len(result_vggt.pts3d):>10,} {len(pts3d_vggt):>10,} {conf_vggt_mean:>10.3f} {conf_vggt_std:>10.3f}")
print(f"{'MapAnything':< {col}} {len(result_ma.pts3d):>10,} {len(pts3d_ma):>10,} {conf_ma_mean:>10.3f} {conf_ma_std:>10.3f}")
print(f"{'VGGT-Omega':< {col}} {len(result_omega.pts3d):>10,} {len(pts3d_omega):>10,} {conf_omega_mean:>10.3f} {conf_omega_std:>10.3f}")

Model             Pts raw   Pts filt  Conf mean   Conf std
--------------------------------------------------------
VGGT-X            500,000    171,665      2.334      1.276
MapAnything       500,000    467,019      0.408      0.391


## §11 — Camera Pose Overlay

Overlays camera frustums from all three models in a single scene — blue for VGGT-X, orange for MapAnything, green for VGGT-Omega. Alignment between the three sets indicates consistent global pose estimation.

In [11]:
pl = pv.Plotter()
add_camera_frustums(pl, result_vggt.extrinsics, color="cornflowerblue")
add_camera_frustums(pl, result_ma.extrinsics, color="darkorange")
add_camera_frustums(pl, result_omega.extrinsics, color="mediumseagreen")
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:33539/index.html?ui=P_0x7172d21b79d0_2&reconnect=auto" class="pyvi…